# Cloud Detection Inference Demo

End-to-end demonstration of the cloud detector on a calibrated L1 img16 Zarr store.

**Pipeline:** L1 Zarr → `calibrate_img` → `predict_cloud_score` → visualizations

**Model:** `cloud_detector_v1` — binary CNN classifier (clear / cloudy),  
trained on 8 Lick Observatory observing runs (2023–2024).

---

### Prerequisites

You need an L1 img16 Zarr store. To generate one from the raw PFF data:

```bash
# Ingest the cloudy batch-9 run (2024-04-19, "Mostly cloudy")
nextflow run . -profile laptop \
    --steps ingest \
    --input_obs_dir /mnt/beegfs/data/L0/obs_Lick.start_2024-04-19T05:39:33Z.runtype_eng-test.pffd \
    --outdir ml/cloud-detection/data/cloudy_demo
```

Then set `L1_STORE` below to one of the resulting `.zarr` paths.

Alternatively, use any existing L1 img16 store on BeeGFS.

In [ ]:
# panoseti_analysis is installed editable (uv sync) — import it directly, no sys.path hacks.
%load_ext autoreload
%autoreload 2


from datetime import UTC, datetime
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import torch

from panoseti_analysis.algorithms.calibrate_img import ImgCalibParams, calibrate_img
from panoseti_analysis.algorithms.cloud_detector import predict_cloud_score
from panoseti_analysis.config.models import CloudInferParams
from panoseti_analysis.io.models import load_classifier
from panoseti_analysis.io.stores import open_store
from panoseti_analysis.paths import ML, MODELS

plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

## §0 Configuration

In [ ]:
# ── Set these paths ──────────────────────────────────────────────────────────
# Default: the bundled cloudy demo run under ml/cloud-detection/data/.
# Override L1_STORE with any L1 img16 .zarr (e.g. on BeeGFS) for a real run:
#   L1_STORE = Path("/mnt/beegfs/runs/my_run/L1/obs.dp_img16.module_1.L1.zarr")
L1_STORE = ML / "cloud-detection" / "data" / "cloudy_demo" / "L1"

MODEL_PATH = MODELS / "cloud_detector_v1.pt"
THRESHOLD = 0.5  # cloud_score ≥ threshold → cloudy
CADENCE_S = 60.0  # inference window (seconds)

# If L1_STORE is a directory (not a .zarr), pick the first img16 store found under it.
if L1_STORE.is_dir() and not L1_STORE.name.endswith(".zarr"):
    candidates = sorted(L1_STORE.rglob("*.dp_img16.*.L1.zarr"))
    if not candidates:
        raise FileNotFoundError(f"No L1 img16 store found under {L1_STORE}")
    L1_STORE = candidates[0]
    print(f"Using: {L1_STORE}")

## §1 Load L1 Zarr

In [ ]:
ds_l1_raw = open_store(L1_STORE)
display(ds_l1_raw)
print()

# Time range
t_ns = ds_l1_raw["unix_t_ns"].values
t_start = datetime.fromtimestamp(t_ns[0] / 1e9, tz=UTC)
t_end = datetime.fromtimestamp(t_ns[-1] / 1e9, tz=UTC)
print(f"Time range : {t_start:%Y-%m-%d %H:%M:%S} → {t_end:%H:%M:%S} UTC")
print(f"Frames     : {len(t_ns):,}")
print(
    f"Duration   : {(t_ns[-1] - t_ns[0]) / 1e9:.0f} s ({(t_ns[-1] - t_ns[0]) / 1e9 / 60:.1f} min)"
)
print()
# Provenance
history = ds_l1_raw.attrs.get("processing_history", [])
for step in history:
    if isinstance(step, dict):
        print(f"  [{step.get('step_name')}] @ {step.get('timestamp_utc', '?')[:16]}")

# Data contract check — validate() raises with a precise error if a required var is missing.
# Pipeline-produced L1 stores always pass; helpful catch when pointing at an ad-hoc store.
if ds_l1_raw.pano.level == "L1":
    ds_l1_raw.pano.validate(level="L1", kind="img")
    print(f"\n✓ pano.level={ds_l1_raw.pano.level!r}  pano.kind={ds_l1_raw.pano.kind!r}")
    qc = ds_l1_raw.attrs.get("qc", {})
    if qc:
        flag = "✓" if qc["isgood"] else "✗"
        print(f"  QC {flag} isgood={qc['isgood']}  metrics={qc['metrics']}")
    else:
        print("  (no QC attrs — store may predate the spring-cleaning refactor)")

## §2 Calibrate

In [ ]:
# calibrate_img expects ds['images'] (T, H, W) uint16/int16
# L1 stores from the pipeline already have 'median_subtracted' — skip calibration.
if "median_subtracted" in ds_l1_raw.data_vars:
    ds_l1 = ds_l1_raw
    print("Store already calibrated (has 'median_subtracted').")
    # Validate the L1 contract before inference — catches ad-hoc stores early.
    ds_l1.pano.validate(level="L1", kind="img")
    print(f"✓ L1/img contract satisfied  ({len(ds_l1['unix_t_ns'])} frames)")
else:
    from panoseti_analysis.algorithms.calibrate_img import ImgCalibParams, calibrate_img

    calib_params = ImgCalibParams()
    ds_l1 = calibrate_img(ds_l1_raw, calib_params)
    print(f"Calibrated: median_subtracted shape = {ds_l1['median_subtracted'].shape}")
    print(f"  Hot pixel mask coverage: {float(ds_l1['hot_pixel_mask'].mean()):.2%}")
    print(f"  Dead pixel mask coverage: {float(ds_l1['dead_pixel_mask'].mean()):.2%}")

## §3 Run Cloud Detection Inference

`predict_cloud_score` lazily gathers **only** the frames each cadence window references
(`da.isel(time=needed)`), so scoring a whole night touches a few thousand frames instead
of materialising the entire multi-GB `median_subtracted` array (this store is 57 GB / 13.6 M
frames, but only ~228 windows are scored). The `%%time` below makes that cost visible.

In [ ]:
%%time
model, bundle = load_classifier(MODEL_PATH)
model.eval()
print(f"Model: {bundle.model_name} v{bundle.model_version}")
print(f"Input spec: {bundle.input_spec}")
print(f"Checksum: {bundle.checksum[:24]}…")
print()

params = CloudInferParams(cadence_s=CADENCE_S, threshold=THRESHOLD)
with torch.no_grad():
    ds_l2 = predict_cloud_score(ds_l1, model, params)

scores = ds_l2["cloud_score"].values
labels = ds_l2["cloud_label"].values
print(f"Windows scored : {len(scores)}")
print(f"cloud_score    : min={scores.min():.3f}  mean={scores.mean():.3f}  max={scores.max():.3f}")
print(f"cloudy windows : {labels.sum()} / {len(labels)} ({labels.mean():.1%})")

## §4 Cloud Score Timeline

In [ ]:
t_l2_ns = ds_l2["unix_t_ns"].values
t_l2_dt = [datetime.fromtimestamp(t / 1e9, tz=UTC) for t in t_l2_ns]

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(t_l2_dt, scores, color="steelblue", linewidth=1.2, zorder=2, label="cloud_score")
ax.axhline(THRESHOLD, color="tomato", linestyle="--", linewidth=1, label=f"threshold={THRESHOLD}")

# Shade cloudy windows
cloudy_mask = labels.astype(bool)
for i, (t, is_cloudy) in enumerate(zip(t_l2_dt, cloudy_mask, strict=False)):
    if is_cloudy:
        ax.axvspan(t_l2_dt[max(0, i - 1)], t, alpha=0.15, color="tomato", zorder=1)

ax.set_ylim(-0.05, 1.05)
ax.set_ylabel("Cloud score")
ax.set_xlabel("UTC time")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax.set_title(f"Cloud detection — {L1_STORE.name}")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Fraction of night classified as cloudy: {labels.mean():.1%}")

## §5 Feature Visualization (4-panel)

For each flagged window, show the four feature maps used by the model.

In [ ]:
# Feature maps for flagged windows. These reuse the features predict_cloud_score already
# returned (ds_l2["feature_raw_fft"] / ["feature_deriv_fft"]) — so there's no re-extraction
# (fast), and no truncated-window artefact in the derivative channel (the old version
# recomputed features on a 60 s slice whose first sample has a degenerate prev≈curr diff).


def _nearest_frame(ds_l1_in, t_center_ns):
    """Lazily load the single median_subtracted frame nearest t_center_ns."""
    t = ds_l1_in["unix_t_ns"].values
    j = int(np.searchsorted(t, t_center_ns).clip(0, len(t) - 1))
    return ds_l1_in["median_subtracted"].isel(time=j).values


def _plot_window(ds_l1_in, ds_l2_in, idx, ax_row, title=""):
    """4-panel display for the scored window at integer index idx."""
    raw_fft = ds_l2_in["feature_raw_fft"].isel(T_l2=idx).values
    deriv_fft = ds_l2_in["feature_deriv_fft"].isel(T_l2=idx).values
    raw_img = _nearest_frame(ds_l1_in, int(ds_l2_in["unix_t_ns"].values[idx]))

    kw = dict(cmap="viridis", interpolation="nearest")
    ax_row[0].imshow(raw_img, **kw)
    ax_row[0].set_title("median_subtracted (current)")
    ax_row[1].imshow(raw_fft, **kw)
    ax_row[1].set_title("raw_fft (ch1)")
    ax_row[2].imshow(deriv_fft, **kw)
    ax_row[2].set_title("deriv_fft (ch0, 60s)")
    ax_row[3].text(
        0.5,
        0.5,
        title,
        ha="center",
        va="center",
        fontsize=9,
        transform=ax_row[3].transAxes,
        wrap=True,
    )
    for ax in ax_row:
        ax.axis("off")


# Show up to 5 flagged (cloudy) windows; fall back to the first few if none flagged.
flagged_idx = np.random.choice(np.where(cloudy_mask)[0], 5)
if len(flagged_idx) == 0:
    print("No cloudy windows detected — showing the first few windows instead.")
    flagged_idx = np.arange(min(4, len(t_l2_ns)))

fig, axes = plt.subplots(len(flagged_idx), 4, figsize=(13, 3 * len(flagged_idx)))
if len(flagged_idx) == 1:
    axes = [axes]

for row_ax, idx in zip(axes, flagged_idx, strict=False):
    t_c = int(t_l2_ns[idx])
    t_label = datetime.fromtimestamp(t_c / 1e9, tz=UTC).strftime("%H:%M:%S")
    title = f"t={t_label}\nscore={float(scores[idx]):.3f}\n{'CLOUDY' if labels[idx] else 'CLEAR'}"
    _plot_window(ds_l1, ds_l2, int(idx), row_ax, title)

plt.suptitle("Feature maps for flagged windows", y=1.01)
plt.tight_layout()
plt.show()

## §6 Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

ax = axes[0]
ax.hist(scores, bins=30, color="steelblue", edgecolor="white", linewidth=0.5)
ax.axvline(THRESHOLD, color="tomato", linestyle="--", label=f"threshold={THRESHOLD}")
ax.set_xlabel("Cloud score")
ax.set_ylabel("Count")
ax.set_title("Score distribution")
ax.legend()

ax = axes[1]
clear_scores = scores[labels == 0]
cloudy_scores = scores[labels == 1]
ax.hist(clear_scores, bins=20, alpha=0.6, color="steelblue", label=f"clear (n={len(clear_scores)})")
ax.hist(cloudy_scores, bins=20, alpha=0.6, color="tomato", label=f"cloudy (n={len(cloudy_scores)})")
ax.set_xlabel("Cloud score")
ax.set_title("Scores by predicted class")
ax.legend()

plt.tight_layout()
plt.show()

## §7 Summary

In [ ]:
duration_min = (t_ns[-1] - t_ns[0]) / 1e9 / 60
print("Run summary")
print(f"  Store       : {L1_STORE.name}")
print(f"  Duration    : {duration_min:.1f} min ({len(t_ns):,} frames)")
print(f"  Windows     : {len(scores)} × {CADENCE_S:.0f}s")
print(f"  Cloudy      : {labels.sum()} windows ({labels.mean():.1%})")
print(f"  Score range : {scores.min():.3f} – {scores.max():.3f}")
print(f"  Model       : {bundle.model_name} v{bundle.model_version}")

## §8 (Optional) Serve inference via Ray Serve

The local path above is already fast (lazy frame gather), so Ray Serve isn't needed for a
one-off demo. It pays off when scoring **many** stores against a **warm GPU replica** — the
same `CloudInferDeployment` the streaming pipeline (`pa-stream-cloud`) uses. This needs a
running Ray cluster with a GPU node (e.g. RAL), so it's gated behind a flag, off by default.

In [ ]:
# Set True only on a Ray cluster with a GPU node (e.g. RAL). Off by default.
USE_RAY_SERVE = False

if USE_RAY_SERVE:
    from panoseti_analysis.adapters.ray.launcher import init_ray
    from panoseti_analysis.adapters.stream.serve_app import deploy_cloud_infer

    init_ray("attach")  # attach to the externally-owned cluster
    handle = deploy_cloud_infer(str(MODEL_PATH), num_replicas=2)

    # infer() is async; .result() resolves the Serve DeploymentResponse from a sync cell.
    ds_l2_serve = await handle.infer.remote(ds_l1, params)
    s = ds_l2_serve["cloud_score"].values
    print(f"Ray Serve scored {len(s)} windows — mean cloud_score {s.mean():.3f}")
else:
    print("Ray Serve cell skipped (USE_RAY_SERVE=False). Set True on a GPU cluster to enable.")